    Bernacchia Alessia, Pioda Tommaso, Villani Giacomo
    Data Project and Hackaton 3
    _Sandra Mitròvich_

In [1]:
import pandas as pd
import numpy as np
import os
import re
import glob
from datetime import datetime
import pyarrow.parquet as pq
from tqdm.auto import tqdm
tqdm.pandas()

os.makedirs("data", exist_ok=True)

# path to the Parquet files folder
PARQUET_PATH = "data/parquet"

# Values that should be treated as NaN/None
NULL_PLACEHOLDERS = ['None', 'nan', 'N/A', '', '-', '/', '?', 'null']

# path where safe utils
AUTH_REGISTRY_PATH = "data/authors_registry.parquet"
RAW_AUTH_REGISTRY_PATH = "data/authors_registry_full.parquet"

In [2]:
# extract parquet files
parquet_folder = os.path.abspath(PARQUET_PATH)
print(f'The parquet files are saved in : {parquet_folder}')
parquet_files = [f for f in os.listdir(parquet_folder) if f.endswith('.parquet')]

print(f"Total Parquet files found: {len(parquet_files)}")


# function to obtain the path of a file given the index
parquet_file_path_n = lambda n: os.path.join(parquet_folder, parquet_files[n])
# function to convert a parquet to a table or to a pandas dataframe
parquet_to_table = lambda parquet_path: pq.read_table(parquet_path)
parquet_to_df = lambda parquet_path: pq.read_table(parquet_path).to_pandas()

# efficiently peek at just the first N rows of a file
def peek_file(n, rows=5):
    path = parquet_file_path_n(n)
    return parquet_to_table(path).slice(0, rows).to_pandas()

# load the full schema/columns only (to verify consistency)
def get_schema(n=0):
    return pq.read_schema(parquet_file_path_n(n))

def stream_all_data():
    """Generator to yield one DataFrame at a time from all parquet files."""
    for i in range(len(parquet_files)):
        yield pd.read_parquet(parquet_file_path_n(i))

The parquet files are saved in : c:\Users\ivan.bernacchia\OneDrive - TINEXT SA\Documents\_Alessia\M-P6203E-DataProjects-Hackaton3_P1\data\parquet
Total Parquet files found: 68


In [3]:
class DBLP_Loader:
    """ Loader class to handle the streaming and processing of the DBLP dataset.
    - Initializes with the folder path containing the parquet files.
    - Provides methods to load all data at once (with caution) or to stream and process files one by one.
    - Includes a specific method to fill missing author details using an authors registry.
    - Offers a safe checkpointing method to save processed data in chunks to avoid memory issues."""
    def __init__(self, folder_path):
        self.folder = os.path.abspath(folder_path)
        
        self.files = sorted([f for f in os.listdir(self.folder) if f.endswith('.parquet')])
        self.current_data = []

    def get_full_df(self):
        """Loads all files into one DataFrame (Careful with RAM!)"""
        all_chunks = []
        for f in tqdm(self.files, desc="Loading Parquet Files"):
            path = os.path.join(self.folder, f)
            all_chunks.append(pd.read_parquet(path))
        return pd.concat(all_chunks, ignore_index=True)
    
    def stream_and_process(self, process_func, update_internal=True):
        """Processes files one by one and optionally updates internal state."""
        results = []

        for f in tqdm(self.files, desc="Processing Batches"):
            path = os.path.join(self.folder, f)
            df = pd.read_parquet(path)
            processed_df = process_func(df)
            results.append(processed_df)

        if update_internal == True:
            self.current_data = results

        return results
    
    def fill_author_gaps(self, author_registry, chunks=None, update_internal=True):
        """
        Specific process function to fill missing author details.
        - Fills missing `id` from `official_name`.
        - Fills missing `official_name` from `id`.
        - Fills missing `org` from `id` and `year`.
        """
        chunks = chunks if chunks is not None else self.current_data  
        reg = author_registry.copy()

        # mapping
        print("Creating mapping dictionaries from the authors registry...")

        print("\tname to official name mapping...")
        # name -> official_name
        # name_to_official_name = reg.explode('name').set_index('name')['official_name'].to_dict()
        name_exploded = reg.explode('name', ignore_index=True)
        name_to_official_name = dict(zip(name_exploded['name'].values, name_exploded['official_name'].values))
        
        print("\tid to official name mapping...")
        # id -> official_name
        # name_to_id = reg.set_index('official_name')['id'].to_dict()
        name_to_id = dict(zip(reg['official_name'].values, reg['id'].values))

        print("\tofficial name to id mapping...")
        # official_name -> id
        # id_to_name = reg.set_index('id')['official_name'].to_dict()
        id_to_name = dict(zip(reg['id'].values, reg['official_name'].values))

        print("\tid-year to org mapping...")
        # id, year -> org
        def valid_format_org_year(cell):
            if isinstance(cell, dict) and 'org' in cell and 'year' in cell:
                return True  # Wrap in a list to standardize the format
            else:
                if isinstance(cell, list) and len(cell) == 2 and isinstance(cell[0], str) and isinstance(cell[1], (int, float)):
                    print(f"Warning: 'org_year' is a list instead of dict. Converting to dict format for consistency.")
                return False  # Invalid format, will be filtered out later

        registry_exploded = reg.explode('org_year', ignore_index=True)
        # filter out rows where 'org_year' is not a valid dict
        mask_valid = registry_exploded['org_year'].map(valid_format_org_year)
        registry_exploded = registry_exploded[mask_valid]
        # divide 'org_year' in two different columns, 'org' and 'year'
        org_year_df = pd.json_normalize(registry_exploded['org_year']) # json normalization is faster
        registry_exploded = pd.concat([registry_exploded[['id']], org_year_df[['org', 'year']]]
                                      , axis=1).dropna(subset=['org', 'year', 'id'])

        # convert types for memory efficiency
        registry_exploded['year'] = registry_exploded['year'].astype('int32', errors='ignore')
        registry_exploded['org'] = registry_exploded['org'].astype('category')


        id_year_to_org = dict(zip(zip(registry_exploded['id'].values, registry_exploded['year'].values),
                                  registry_exploded['org'].values))
        
        del reg, org_year_df, registry_exploded, name_exploded  # free memory

        # function to fill the gaps in the authors data
        def fill_missing(authors, year):
            if authors is None or not isinstance(authors, list) or len(authors) == 0:
                return authors  # No authors to process, skip to next row
            
            for a in authors:
                if a is None or pd.isna(a) or not isinstance(a, dict):
                    continue  # Skip invalid author entries
                
                name = a.get('name')
                a_id = a.get('id')

                if name and name in name_to_official_name:
                    official_name = name_to_official_name.get(name)
                    a['name'] = official_name

                    # no ID, but official name is known
                    if pd.isna(a_id) and official_name in name_to_id:
                        a['id'] = name_to_id.get(official_name)

                # no name, but ID is known
                elif pd.isna(name) and a_id in id_to_name:
                    a['name'] = id_to_name.get(a_id)

                # not org, but ID and year are known
                if pd.isna(a.get('org')) and a_id and isinstance(year, (int, float)) and (a_id, year) in id_year_to_org:
                    a['org'] = id_year_to_org.get((a_id, year))

            return authors
        
        # process chunks
        results = []
        for c in tqdm(chunks, desc="Filling Authors Gaps Chunks:"):
            c = c.copy()
            # fill missing values
            c['authors'] = [fill_missing(authors, year) for authors, year in zip(c['authors'], c['year'])]
            
            results.append(c)

        if update_internal == True:
            self.current_data = results

        return results
    
    def fix_venue_mismatch(self, chunks=None, update_internal=True):
        """ Checks for mismatches between 'venue' and 'doc_type' and attempts to fix them based on simple heuristics in every chunk."""
        chunks = chunks if chunks is not None else self.current_data
        results = []

        n_conf_as_journ = 0
        n_journ_as_conf = 0
        
        def check_and_fix_venue_mismatch(df):
            """ Checks for mismatches between 'venue' and 'doc_type' and attempts to fix them based on simple heuristics."""
            # Example: If 'Conference' in venue name but 'Journal' in doc_type, set doc_type to 'Conference'
            conf_as_journ = (df['venue'].str.contains('Conference', case=False, na=False)) & (df['doc_type'].str.contains('Journal', case=False, na=False))
            journ_as_conf = (df['venue'].str.contains('Journal', case=False, na=False)) & (df['doc_type'].str.contains('Conference', case=False, na=False))

            n_conf_as_journ = conf_as_journ.sum()
            n_journ_as_conf = journ_as_conf.sum()
            
            df.loc[conf_as_journ, 'doc_type'] = 'Conference'
            df.loc[journ_as_conf, 'doc_type'] = 'Journal'

            return df, n_conf_as_journ, n_journ_as_conf
        
        for c in tqdm(chunks, desc="Fixing Venue Mismatch Chunks:"):
            c = c.copy()
            c, n1, n2 = check_and_fix_venue_mismatch(c)
            
            n_conf_as_journ += n1
            n_journ_as_conf += n2
            
            results.append(c)
        
        if update_internal == True:
            self.current_data = results

        print(f"Found {n_conf_as_journ} mismatches where 'Conference' is in venue but doc_type is 'Journal'.")
        print(f"Found {n_journ_as_conf} mismatches where 'Journal' is in venue but doc_type is 'Conference'.")

        return results
    
    def safe_checkpoint(self, base_name='01_processed_data', path='data/checkpoints', chunk_size = 100000):
        """
        Save the processed data to Parquet files in chunks.
        Parameters:
            - data: DataFrame to save
            - path: Directory where to save the Parquet files
            - base_name: The base name for the saved files
        """
        if not os.path.exists(path):
            os.makedirs(path)

        data = pd.concat(self.current_data, ignore_index=True)
        # Split data into smaller chunks and save each as a Parquet file
        total_chunks = len(data) // chunk_size + (len(data) % chunk_size > 0)
        
        for i in tqdm(range(total_chunks), desc="Saving Parquet Files"):
            start_idx = i * chunk_size
            end_idx = min((i + 1) * chunk_size, len(data))
            
            chunk = data.iloc[start_idx:end_idx]
            file_path = os.path.join(path, f"{base_name}_part_{i + 1}.parquet")
            chunk.to_parquet(file_path, engine='pyarrow')

        print(f"Checkpoint saved to {path}")

In [4]:
loader = DBLP_Loader(PARQUET_PATH)
print(f"Ready to process {len(loader.files)} files.")

Ready to process 68 files.


# A - Exploration and Cleaning of the Dataset

## Overview: DBLP Citation Network


The dataset is a comprehensive collection of scientific papers sourced from the DBLP (Computer Science Bibliography). 

It is designed for tasks involving data exploration, citation prediction, and classification within the academic domain.

Source: [DBLP (Computer Science Bibliography)](https://opendata.aminer.cn/dataset/DBLP-Citation-network-V18.zip)

Size: 6.7 million papers

Format: JSONL (JSON Lines) file

### Key Data Fields:

In [5]:
# column groups based on categories
NUMERIC_COLS = ['year', 'n_citation', 'page_start', 'page_end', 'volume', 'issue']
STRING_COLS = ['id', 'title', 'lang', 'doc_type', 'venue', 'issn', 'isbn', 'doi', 'abstract']
ARRAY_OF_STR_COLS = ['references', 'keywords', 'url']
ARRAY_OF_DICT_COLS = ['authors']

- **id** (_string_): unique identifier for the paper, mix of alpha characters and numbers.
- **title** (_string_): title of the research paper.
- **year** (_integer_): year of publication.
- **authors** (_list_): including author name, id, and affiliation (org).
- **lang** (_string_): detected language of the text.
- **doc_type** (_string_ [conference, journal]): categorization one between `conference` or `journal`.
- **venue** (_string_): the conference or journal where it was published.
- **references** (_list_): paper IDs cited by this document.
- **keywords** (_list_): relevant tags or index terms.
- **abstract** (_string_): summary of the paper's content.
- **n_citation** (_integer_): number of received citations.
---
- **page_start** (_integer_): starting page number in publication.
- **page_end** (_integer_): ending page number in publication.
- **volume** (_integer_): volume number of the publication.
- **issue** (_integer_): issue number of the publication.
- **issn** (_string_): International Standard Serial Number.
- **isbn** (_string_): International Standard Book Number.
- **doi** (_string_): Digital Object Identifier link, mix of alpha characters and numbers.
- **url** (_list_): link to external web resources.

## Exploration

In [6]:
# extract files
# tab_0 = parquet_to_table(parquet_file_path_n(0))
# df_0 = tab_0.slice(0, 3).to_pandas()
# df_0.head(3)
peek_file(0, rows=3)

,id,title,abstract,keywords,year,authors,references,page_start,page_end,lang,volume,issue,issn,isbn,doi,url,n_citation,venue,doc_type
0,5390877920f70186a0d2ce7f,Top-Down Construction of 3-D Mechanical Object...,,"[First Page, 3-D Mechanical Object Shapes, Eng...",1984,"[{'id': '62aad3b2d9f2040d085dc9f9', 'name': 'H...","[5390962020f70186a0df3bf2, 5390879d20f70186a0d...",32,40,en,17,12,0018-9162,,10.1109/mc.1984.1659026,[],46,Computer,Journal
1,5390877920f70186a0d2d85b,MH: A Multifarious User Agent.,The UCI version of the Rand Message Handling S...,"[Computer network mail, Distributed mail syste...",1985,"[{'id': '', 'name': 'MT ROSE', 'org': None}, {...",[5390880720f70186a0d7a97e],65,80,en,10,2,0169-7552,,10.1016/0169-7552(85)90077-7,[https://doi.org/10.1016/0169-7552(85)90077-7],4,Computer Networks and ISDN Systems,Journal
2,5390877920f70186a0d2e433,Characterizing an Optimal Input in Perturbed C...,When every feasible stable perturbation of dat...,"[Optimal input, region of stability, input con...",1983,"[{'id': '53f43a54dabfaee43ec54f70', 'name': 'S...","[53e99d28b7602d97025da4e6, 5390877920f70186a0d...",109,121,en,25,1,0025-5610,,10.1007/bf02591721,[https://dl.acm.org/doi/10.5555/9504.9513],20,Mathematical Programming,Journal


### Data Quality
Validate types and do a first cleaning:
- set all NaNs
- check the year is in a valid range
- check the doc_type has a valid category

In [7]:
def validate_and_clean_types(df):
    """
    Validates types, cleans strings, and checks year constraints.
    """
    df = df.copy()

    # validate string and handle placeholder as NaN
    for col in STRING_COLS:
        if col in df.columns:
            df[col] = df[col].astype(str)
            # replace placeholder strings with actual np.nan
            df[col] = df[col].replace(NULL_PLACEHOLDERS, np.nan)
    
    # validate Numeric Columns (errors='coerce' turns bad data to NaN)
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            # not smaller than 0
            df.loc[(df[col] <= 0), col] = 0

    # validate Publication Year
    current_year = datetime.now().year
    # year > current year or <= 0 -> set to NaN
    if 'year' in df.columns:
        df.loc[(df['year'] > current_year) | (df['year'] <= 1800), 'year'] = np.nan

    # normalize categorical
    if 'doc_type' in df.columns:
        # standardize to lowercase 'conference' or 'journal'
        df['doc_type'] = df['doc_type'].str.lower()
        # ensure it only contains the allowed categories
        valid_types = ['conference', 'journal']
        df.loc[~df['doc_type'].isin(valid_types), 'doc_type'] = np.nan

    # validate adn handle lists columns
    for col in ARRAY_OF_STR_COLS:
        if col in df.columns:
            # ensure None/NaN instead of empty lists and ensure they are valid list types
            df[col] = df[col].apply(lambda x: x if isinstance(x, np.ndarray) and len(x) > 0 and x.dtype == 'O' and all(isinstance(i, str) for i in x) else np.nan)

    # validate and handle array of dictionaries columns
    for col in ARRAY_OF_DICT_COLS:
        if col in df.columns:
            # ensure None/NaN instead of empty lists and ensure they are valid list types
            df[col] = df[col].apply(lambda x: x if isinstance(x, np.ndarray) and len(x) > 0 and x.dtype == 'O' and all(isinstance(i, dict) for i in x) else np.nan)

    return df

In [8]:
cleaned_chunks = loader.stream_and_process(validate_and_clean_types)

if cleaned_chunks:
    print("Quality Check Sample (First 5 rows of first chunk):")
    display(cleaned_chunks[0].head(3))
    
    # count how many years were invalidated (converted to NaN)
    nan_years = cleaned_chunks[0]['year'].isna().sum()
    print(f"\nMissing or Invalid years in Chunk 0: {nan_years}")

Processing Batches:   0%|          | 0/68 [00:00<?, ?it/s]

Quality Check Sample (First 5 rows of first chunk):


,id,title,abstract,keywords,year,authors,references,page_start,page_end,lang,volume,issue,issn,isbn,doi,url,n_citation,venue,doc_type
0,5390877920f70186a0d2ce7f,Top-Down Construction of 3-D Mechanical Object...,NaN,"[First Page, 3-D Mechanical Object Shapes, Eng...",1984.0,"[{'id': '62aad3b2d9f2040d085dc9f9', 'name': 'H...","[5390962020f70186a0df3bf2, 5390879d20f70186a0d...",32.0,40.0,en,17.0,12.0,0018-9162,NaN,10.1109/mc.1984.1659026,NaN,46,Computer,journal
1,5390877920f70186a0d2d85b,MH: A Multifarious User Agent.,The UCI version of the Rand Message Handling S...,"[Computer network mail, Distributed mail syste...",1985.0,"[{'id': '', 'name': 'MT ROSE', 'org': None}, {...",[5390880720f70186a0d7a97e],65.0,80.0,en,10.0,2.0,0169-7552,NaN,10.1016/0169-7552(85)90077-7,[https://doi.org/10.1016/0169-7552(85)90077-7],4,Computer Networks and ISDN Systems,journal
2,5390877920f70186a0d2e433,Characterizing an Optimal Input in Perturbed C...,When every feasible stable perturbation of dat...,"[Optimal input, region of stability, input con...",1983.0,"[{'id': '53f43a54dabfaee43ec54f70', 'name': 'S...","[53e99d28b7602d97025da4e6, 5390877920f70186a0d...",109.0,121.0,en,25.0,1.0,0025-5610,NaN,10.1007/bf02591721,[https://dl.acm.org/doi/10.5555/9504.9513],20,Mathematical Programming,journal



Missing or Invalid years in Chunk 0: 1


### Compile Authors registry
Create and compile a df with all authors and them info

In [9]:
def tmp_authors_registry_chunks(authors_registry, chunk_size = 500_000, output_dir = "data/authors_registry_chunks"):
    """Safe checkpoints to safe the raw data of authors registry"""
    os.makedirs(output_dir, exist_ok=True)

    def convert_row(x):
        return list(x) if isinstance(x, (set, list)) else x

    def convert_org_year(x):
        if isinstance(x, (set, list)):
            return [
                [org, int(year) if pd.notna(year) else None]
                for org, year in x
            ]
        return x

    for i in tqdm(range(0, len(authors_registry), chunk_size), desc="Processing chunks"):
        chunk_id = i // chunk_size
        chunk_path = f"{output_dir}/part_{chunk_id}.parquet"

        if os.path.exists(chunk_path):
            continue

        chunk = authors_registry.iloc[i:i+chunk_size].copy()

        # conversions
        for col in ['name', 'keywords', 'lang']:
            chunk[col] = [convert_row(x) for x in chunk[col]]

        chunk['org_year'] = [convert_org_year(x) for x in chunk['org_year']]

        # safe
        chunk.to_parquet(chunk_path)

def tmp_authors_registry_merge_chunks(data_dir = "data/authors_registry_chunks", output_dir = "data/authors_registry_full.parquet"):
    """merge in an unique parquet file"""
    files = sorted(glob.glob(f"{data_dir}/part_*.parquet"))

    writer = None

    for f in tqdm(files, desc="Merging chunks"):
        table = pq.read_table(f)

        if writer is None:
            writer = pq.ParquetWriter(
                output_dir,
                table.schema
            )

        writer.write_table(table)

    if writer:
        writer.close()

    print("Merge completed!")

def safe_authors_registry_raw_checkpoint(authors_registry, chunk_size = 500_000, chunks_dir="data/authors_registry_chunks", output_dir="data/authors_registry_full.parquet"):
    tmp_authors_registry_chunks(authors_registry, chunk_size=chunk_size, output_dir=chunks_dir)
    tmp_authors_registry_merge_chunks(data_dir=chunks_dir, output_dir=output_dir)

In [10]:
from collections import Counter 
import ast

def is_valid_name(name):
    # ^ : start of the string
    # [A-Za-z\s]+ : one or more letters (upper or lower case) or whitespace characters
    # $ : end of the string
    return bool(re.match(r'^[A-Za-z\s]+$', name))

def union_sets(series):
        return set().union(*series)

def union_org_year(series):
    unique_orgs = set()
    for sublist in series:
        if isinstance(sublist, (list, set)):
            for item in sublist:
                if isinstance(item, (list, tuple)):
                    unique_orgs.add(item)
                elif isinstance(item, dict) and 'org' in item and 'year' in item:
                    unique_orgs.add((item['org'], item['year']))
    return list(unique_orgs.values())

def extract_author_info(df):
    """
    Extract authors' info from a chunk of papers
    """
    author_records = []

    for _, row in df.iterrows():
        authors_list = row.get('authors')
        # print(f'debug 1: {len(authors_list)}')
        
        # if not isinstance(authors_list, list):
        #     continue

        if isinstance(authors_list, str):
            try:
                authors_list = ast.literal_eval(authors_list)
            except:
                continue
            
        keywords = set(row.get('keywords')) if isinstance(row.get('keywords'), list) else set()
        lang = row.get('lang')
        year = row.get('year')
        # print(f'\tdebug 2: {lang}, {year}')

        for auth in authors_list:
            auth_id = auth.get('id')
            auth_name = auth.get('name')
            auth_org = auth.get('org')
            # print(f'\t\tdebug 3: {auth_id}, {auth_name}, {auth_org}')

            # id is the primary key otherwise the name
            has_id = bool(auth_id and auth_id != '')
            key = auth_id if has_id else auth_name
            if not key: 
                continue
            # print(f'\t\t\tdebug 4:{has_id}, {key}')

            author_records.append({
                'id_or_name': key,
                'is_id': has_id,
                'name': [auth_name] if auth_name else list(),
                'org_year': {(auth_org, year)} if auth_org and not pd.isna(year) else set(),
                'keywords': keywords,
                'lang': {lang} if lang and not pd.isna(lang) else set()
            })

    # create a temp dataframe with chunk's info
    if not author_records:
        return pd.DataFrame()

    temp_df = pd.DataFrame(author_records)
    
    # chunk aggregation
    return temp_df.groupby('id_or_name').agg({
        'name': 'sum',
        'is_id': 'max',
        'org_year': union_org_year,
        'keywords': union_sets,
        'lang': union_sets
    }).reset_index()


def assigns_ids(df):
    """Assign an ID to authors that don't have it, based on their official name."""
    # mapping official_name to ID
    name_to_id = df[df['is_id'] == True].explode('official_name').set_index('official_name')['id_or_name'].to_dict()

    # assign IDs to authors without an ID based on their official name
    def assign_name_to_id(row):
        if not row['is_id'] and isinstance(row['official_name'], str) and row['official_name']:
            if row['official_name'] in name_to_id:
                return name_to_id[row['official_name']]
        return row['id_or_name'] if row['is_id'] else None

    df['id'] = df.apply(assign_name_to_id, axis=1)
    return df

def assign_official_name(df):
    """Assign an official name to each author based on the most common name used."""
    def most_common_name(names):
        if isinstance(names, list) and names:
            return Counter(names).most_common(1)[0][0]
        return None

    df['official_name'] = df['name'].apply(most_common_name)
    return df

def clean_names(df):
    """Clean the names by removing invalid entries."""
    def filter_names(names):
        # keep only valid names (alphabets and spaces)
        return [name for name in names if isinstance(name, str) and is_valid_name(name)]
    
    df['name'] = df['name'].apply(filter_names)
    return df

def clean_org_year(df):
    """Clean the org_year column to ensure it's in the correct format."""
    def clean_org_year_item(org_year):
        if isinstance(org_year, (set, list)):
            cleaned = []
            for item in org_year:
                if isinstance(item, (list, tuple)) and len(item) == 2:
                    org, year = item
                    cleaned.append((org, int(year) if pd.notna(year) else None))
                elif isinstance(item, dict) and 'org' in item and 'year' in item:
                    cleaned.append((item['org'], int(item['year']) if pd.notna(item['year']) else None))
            return cleaned
        return org_year

    df['org_year'] = df['org_year'].apply(clean_org_year_item)
    return df

def safe_authors_registry(df, path=AUTH_REGISTRY_PATH):
    """Save the refined authors registry to a parquet file."""
    df.drop(columns=['is_id', 'id_or_name']).to_parquet(path)
    print(f"Authors registry saved to {path}")    

def refine_authors_df(df):
    """ 
    refine the df of authors:
    - assign an id to the one that don't have it
    - assign an unique name ('official_name') based on the most used one
    and save the df as a parquet
    """
    tqdm.pandas(desc="Refining Authors DataFrame")
    # df = df.pipe(assigns_ids).pipe(assign_official_name).pipe(clean_org_year)
    print("\tCleaning the names...")
    df = clean_names(df)
    print("\tCleaning the organization and year information...")
    df = clean_org_year(df)
    print("\tAssigning official names...")
    df = assign_official_name(df)
    print("\tAssigning IDs...")
    df = assigns_ids(df)

    safe_authors_registry(df)
    return df


In [11]:
if not os.path.exists(AUTH_REGISTRY_PATH):
    if not os.path.exists(RAW_AUTH_REGISTRY_PATH):
        # extract the info
        author_chunks = loader.stream_and_process(extract_author_info, update_internal=False)

        # aggregate info
        print("Authors aggregation in progress...")
        authors_registry = None

        for chunk in author_chunks:
            if chunk.empty:
                continue
            if authors_registry is None:
                authors_registry = chunk
            else:
                authors_registry = pd.concat([authors_registry, chunk], ignore_index=True)
            authors_registry = authors_registry.groupby('id_or_name').agg({
                'is_id': 'max',
                'name': 'sum',
                'org_year': union_org_year,
                'keywords': union_sets,
                'lang': union_sets
            }).reset_index()

            safe_authors_registry_raw_checkpoint(authors_registry=authors_registry)
    else:
        print(f'Read and import file authors raw data from: {RAW_AUTH_REGISTRY_PATH}')
        authors_registry = pd.read_parquet(RAW_AUTH_REGISTRY_PATH)

    print(f"Found {len(authors_registry)} unique authors.")

    # refine the dataframe
    print("Authors refining in progress...")
    final_authors_registry = refine_authors_df(authors_registry)

    # conver set in lists and save it
    reg_to_save = final_authors_registry.copy()

    for col in ['name', 'keywords', 'lang']:
        reg_to_save[col] = [list(x) if isinstance(x, (set, list)) else x for x in reg_to_save[col]]
        # reg_to_save[col].apply(lambda x: list(x) if isinstance(x, (set, list)) else x)

    reg_to_save['org_year'] = [
        [{"org": org, "year": int(year) if pd.notna(year) else None} for org, year in x] 
        if isinstance(x, (set, list)) else x for x in reg_to_save['org_year']
    ]
    reg_to_save.drop(columns=['is_id']).to_parquet(AUTH_REGISTRY_PATH)
else: 
    print(f'Read and import file from: {AUTH_REGISTRY_PATH}')
    final_authors_registry = pd.read_parquet(AUTH_REGISTRY_PATH)

Read and import file from: data/authors_registry.parquet


In [12]:
display(final_authors_registry[final_authors_registry['id'].isna()].head(3))
display(final_authors_registry[final_authors_registry['id'].notna()].head(3))

,id_or_name,name,org_year,keywords,lang,official_name,id
0,\nHelmut Klemm freier Journalist,[\nHelmut Klemm freier Journalist],"[{'org': 'Würzburg', 'year': 2003}]",[],[en],\nHelmut Klemm freier Journalist,NaN
1,,"[ , , , , , , , , , , , , , , , ...",[{'org': 'Defence Science and Technology Group...,[],[en],,NaN
2,,"[ , , , , ]",[{'org': 'Institute of Mathematics University ...,[],[en],,NaN


,id_or_name,name,org_year,keywords,lang,official_name,id
100,Sharmila,[ Sharmila],[],[],[],Sharmila,64036385adc7183bcb77e117
179,53f3186ddabfae9a84425c58,"[H Ancin, Hakan Ancin, H Ancin, H Ancin, Hakan...","[{'org': 'RENSSELAER POLYTECH INST,DEPT ELECT ...",[],[en],H Ancin,53f3186ddabfae9a84425c58
180,53f3186fdabfae9a84425cde,[],"[{'org': 'king abdulaziz university', 'year': ...",[],[en],NaN,53f3186fdabfae9a84425cde


### Assign missing author's id, name and org
- Assign the id to the authors without id, based on the official name assigned.
- Assign the name to the authors without name, based on the id and official name assigned (most used name).
- Assign an org to the authors without org, based on the author id and on the year of publication.

In [13]:
loader.current_data[0].head(3)

,id,title,abstract,keywords,year,authors,references,page_start,page_end,lang,volume,issue,issn,isbn,doi,url,n_citation,venue,doc_type
0,5390877920f70186a0d2ce7f,Top-Down Construction of 3-D Mechanical Object...,NaN,"[First Page, 3-D Mechanical Object Shapes, Eng...",1984.0,"[{'id': '62aad3b2d9f2040d085dc9f9', 'name': 'H...","[5390962020f70186a0df3bf2, 5390879d20f70186a0d...",32.0,40.0,en,17.0,12.0,0018-9162,NaN,10.1109/mc.1984.1659026,NaN,46,Computer,journal
1,5390877920f70186a0d2d85b,MH: A Multifarious User Agent.,The UCI version of the Rand Message Handling S...,"[Computer network mail, Distributed mail syste...",1985.0,"[{'id': '', 'name': 'MT ROSE', 'org': None}, {...",[5390880720f70186a0d7a97e],65.0,80.0,en,10.0,2.0,0169-7552,NaN,10.1016/0169-7552(85)90077-7,[https://doi.org/10.1016/0169-7552(85)90077-7],4,Computer Networks and ISDN Systems,journal
2,5390877920f70186a0d2e433,Characterizing an Optimal Input in Perturbed C...,When every feasible stable perturbation of dat...,"[Optimal input, region of stability, input con...",1983.0,"[{'id': '53f43a54dabfaee43ec54f70', 'name': 'S...","[53e99d28b7602d97025da4e6, 5390877920f70186a0d...",109.0,121.0,en,25.0,1.0,0025-5610,NaN,10.1007/bf02591721,[https://dl.acm.org/doi/10.5555/9504.9513],20,Mathematical Programming,journal


In [14]:
loader.current_data[0].iloc[0:3][['year', 'authors']]

,year,authors
0,1984.0,"[{'id': '62aad3b2d9f2040d085dc9f9', 'name': 'H..."
1,1985.0,"[{'id': '', 'name': 'MT ROSE', 'org': None}, {..."
2,1983.0,"[{'id': '53f43a54dabfaee43ec54f70', 'name': 'S..."


In [15]:
chunks_filled = loader.fill_author_gaps(final_authors_registry)

Creating mapping dictionaries from the authors registry...
	name to official name mapping...
	id to official name mapping...
	official name to id mapping...
	id-year to org mapping...


Filling Authors Gaps Chunks::   0%|          | 0/68 [00:00<?, ?it/s]

### Correct the venue mismatches
- when there is 'Conference' in venue name, but 'Journal' in doc_type
- when there is 'Journal' in venue name, but 'Conference' in doc_type

In [16]:
chunks_cleaned = loader.fix_venue_mismatch(chunks_filled)

Fixing Venue Mismatch Chunks::   0%|          | 0/68 [00:00<?, ?it/s]

Found 83983 mismatches where 'Conference' is in venue but doc_type is 'Journal'.
Found 20086 mismatches where 'Journal' is in venue but doc_type is 'Conference'.


### Save cleaned data